In [8]:
# %pip uninstall -y transformers tokenizers huggingface-hub safetensors
# %pip install "transformers==4.56.1" "tokenizers==0.22.0" "pillow>=10,<11"

# !pip install --upgrade huggingface_hub evaluate
from datasets import load_dataset
from transformers import AutoTokenizer

In [9]:
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

c:\Users\raval\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\raval\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  class HfFileMetadata:


RemoteEntryNotFoundError: 404 Client Error. (Request ID: Root=1-6ac3da14-1315d915501667673bdf7db8;6188dbb6-8de0-497a-aead-8564a6345ce0)

Entry Not Found for url: https://huggingface.co/api/models/distilbert/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false.
additional_chat_templates does not exist on "main"

In [ ]:
dataset = load_dataset('stanfordnlp/imdb')
train_ds = dataset['train'].shuffle(seed=42).select(range(2000))
test_ds = dataset['test'].shuffle(seed=42).select(range(500))

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
print(train_ds[0]['text'][:300])
print(train_ds[0]['label'])

There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot 
1


In [ ]:
def tokenize_batch(batch):
  return tokenizer(
      batch['text'],
      padding=True,
      truncation=True,
      max_length=128
      )

In [ ]:
train_tok = train_ds.map(tokenize_batch, batched=True, batch_size=None)
test_tok = test_ds.map(tokenize_batch, batched=True, batch_size=None)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
train_tok = train_tok.remove_columns(['text'])
test_tok = test_tok.remove_columns(['text'])
train_tok.set_format('torch')
test_tok.set_format('torch')

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = 2)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
args = TrainingArguments(
    output_dir = './bert_imdb_results',
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    learning_rate = 2e-5,
    num_train_epochs = 3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay = 0.01,
    report_to = 'none')

In [ ]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.1 MB/s eta 0:00:00


In [ ]:
import evaluate
import numpy as np

# Load the evaluation metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Get the highest probability logit index as the predicted class
    predictions = np.argmax(predictions, axis=-1)

    # Calculate and return the metric
    return metric.compute(predictions=predictions, references=labels)


In [ ]:
trainer = Trainer(model = model,
                  args = args,
                  train_dataset = train_tok,
                  eval_dataset = test_tok,
                  compute_metrics = compute_metrics)

In [ ]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.476976,0.808000
2,0.393586,0.483362,0.830000
3,0.393586,0.586762,0.824000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=750, training_loss=0.3223279622395833, metrics={'train_runtime': 4614.0373, 'train_samples_per_second': 1.3, 'train_steps_per_second': 0.163, 'total_flos': 198701097984000.0, 'train_loss': 0.3223279622395833, 'epoch': 3.0})

In [ ]:
metrics = trainer.evaluate()
print(metrics)

In [ ]:
sample_text = 'The film is meaningful, emotional beautifully performed.'

inputs = tokenizer(sample_text, padding=True, truncation=True, return_tensors='pt')
#inputs = tokenizer(sample_text, padding=True, truncation=True, return_tensors='pt', max_length = 128)
outputs = model(**inputs)
#predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
predicted_label = outputs.logits.argmax(dim=-1).item()
print('Predicted Sentiment: ', 'Positive' if predicted_label == 1 else 'Negative')
#